In [1]:
import pandas as pd
import pathlib
import subprocess, sys

### extract transportation projects by strategy based on RTP25_ID

In [3]:
# First, export all project coding tasks from Asana project https://app.asana.com/1/11860278793487/project/1204959680579890/list/1205088048625555,
# saved as M:\Application\Model One\RTP2030\StrategyEvaluation\PBA50+_TM_Project_Coding.csv

# load the data
project_df = pd.read_csv(
    r'M:\Application\Model One\RTP2030\StrategyEvaluation\INPUTS\PBA50+_TM_Project_Coding.csv',
    usecols=['Name', 'Section/Column', 'NetworkProjects Folder Name', 'Opening Year', 'In PBA50+', 'RTP25_ID', 'Coding Status']
)
display(project_df.head(3))
display(project_df.loc[project_df['RTP25_ID'].notnull()])

,Name,Section/Column,Coding Status,NetworkProjects Folder Name,Opening Year,In PBA50+,RTP25_ID
0,Follow up on 5934 (190) Bayview Connections,Untitled section,NaN,NaN,NaN,NaN,NaN
1,I880_US101_AdaptiveRampMetering [partial; may ...,Admin,NaN,I880_US101_AdaptiveRampMetering,2023.0,NaN,NaN
2,Develop a framework for systematically increas...,Admin,NaN,NaN,NaN,NaN,NaN


,Name,Section/Column,Coding Status,NetworkProjects Folder Name,Opening Year,In PBA50+,RTP25_ID
232,VTA Eastridge to BART Regional Connector,Under Construction: Add to Baseline (FOR NETWO...,in net spec,MAJ_SCL050009_VTA_Eastridge_Extension,NaN,NoProject,25-T12-179
249,Roadway Widening | Jepson Pkwy-Leisure Town Rd...,Committed but Not Yet Under Construction: Add ...,in net spec,SOL110006_Jepson_1B_1C,2030.0,"NoProject, Exclude from EIR1 (transportation),...",25-T08-129
258,T2: Fare integration (Improve the Rider Experi...,Other Strategy Coding,NaN,NaN,NaN,NaN,25-T02-008
259,T3: Improve the Rider Experience through Refin...,Other Strategy Coding,in net spec,Transform_SeamlessTransit,NaN,"FBP, EIR2 (land use change only)",25-T03-011
278,T5 Parking Pricing Strategy | Regional,Other Strategy Coding,in net spec,NaN,NaN,"FBP, EIR2 (land use change only)",25-T05-014
...,...,...,...,...,...,...,...
580,BART Irvington Station,PPA projects,in net spec,BART_Irvington_Infill,2043.0,"FBP, EIR2 (land use change only)",25-T11-167
581,BART Core Capacity,PPA projects,in net spec,MAJ_REG090037_BART_Core_Cap,2028.0,"FBP, EIR2 (land use change only)",25-T11-173
582,BART Silicon Valley Phase II,PPA projects,in net spec,MAJ_BRT030001_BART_to_SanJose,2037.0,"FBP, EIR2 (land use change only)",25-T12-180
584,Caltrain Portal (formerly Downtown Rail Extens...,PPA projects,in net spec,MAJ_SF_050002_Caltrain_Ext_TransbayTerminal,2036.0,"FBP, EIR2 (land use change only)",25-T12-178


In [4]:
# Blueprint projects coded in TM1 network: has RTP25_ID and NetworkProjects Folder Name
project_blueprint_df = project_df.loc[
    project_df['RTP25_ID'].notnull() & \
    project_df['NetworkProjects Folder Name'].notnull() & \
    (project_df['Coding Status'] != 'No coding needed')]

# projects for strategy T6
projects_T6_df = project_blueprint_df.loc[project_blueprint_df['RTP25_ID'].str.startswith('25-T06-')].copy()
print(projects_T6_df.shape[0], "projects in Strategy T6")
display(projects_T6_df.head(3))

# some projects both '(old)' and '(new)' in NetworkProjects Folder Name, with old names coming from previous network coding;
# clean them up to only keep the new name
has_newline = projects_T6_df['NetworkProjects Folder Name'].str.contains('(', regex=False, na=False)
if has_newline.any():
    print('need to fix the following NetworkProjects Folder Name entries')
    display(projects_T6_df.loc[has_newline])
    projects_T6_df.loc[has_newline, 'NetworkProjects Folder Name'] = (
        projects_T6_df.loc[has_newline, 'NetworkProjects Folder Name'].str.split('\n').str[1]
    )
    projects_T6_df['NetworkProjects Folder Name'] = (
        projects_T6_df['NetworkProjects Folder Name'].str.replace(r' \(new\)$', '', regex=True)
    )
    print('fixed the following NetworkProjects Folder Name entries')
    display(projects_T6_df.loc[has_newline])

# print out the NetworkProjects Folder Name to add to net_spec_blueprint.py
print(projects_T6_df['NetworkProjects Folder Name'].tolist())

33 projects in Strategy T6


,Name,Section/Column,Coding Status,NetworkProjects Folder Name,Opening Year,In PBA50+,RTP25_ID
281,Express Lane | HOV Conversion | SR-85 | Betwee...,Non-Transit Projects - Uncommitted: only in FB...,in net spec,EXP_uncommitted_noAllLaneTolling (old)\nEXP_SC...,2028.0,"FBP, EIR2 (land use change only)",25-T06-065
301,Interchange Improvements | I-880 | Industrial ...,Non-Transit Projects - Uncommitted: only in FB...,in net spec,ALA110002_I880_Industrial_Interchange,2030.0,"FBP, EIR2 (land use change only), Exclude from...",25-T06-043
302,Interchange Improvements | I-80 | Ashby Ave,Non-Transit Projects - Uncommitted: only in FB...,in net spec,ALA170002_I80_Ashby_Int,2030.0,"FBP, EIR2 (land use change only)",25-T06-019


need to fix the following NetworkProjects Folder Name entries


,Name,Section/Column,Coding Status,NetworkProjects Folder Name,Opening Year,In PBA50+,RTP25_ID
281,Express Lane | HOV Conversion | SR-85 | Betwee...,Non-Transit Projects - Uncommitted: only in FB...,in net spec,EXP_uncommitted_noAllLaneTolling (old)\nEXP_SC...,2028.0,"FBP, EIR2 (land use change only)",25-T06-065
339,Interchange Improvements | US-101 | Produce Ave,Non-Transit Projects - Uncommitted: only in FB...,in net spec,STIP_ProduceAve (old)\nSM110003_US101_ProduceA...,2040.0,"FBP, EIR2 (land use change only)",25-T06-054
569,I-80/I-680/SR-12 Interchange,PPA projects,in net spec,MAJ_SOL070020_I80_I680_SR12_Int_2B_7 (old);\nM...,NaN,"FBP, EIR2 (land use change only)",25-T06-024


fixed the following NetworkProjects Folder Name entries


,Name,Section/Column,Coding Status,NetworkProjects Folder Name,Opening Year,In PBA50+,RTP25_ID
281,Express Lane | HOV Conversion | SR-85 | Betwee...,Non-Transit Projects - Uncommitted: only in FB...,in net spec,EXP_SC_SR85_SR87_to_US101_Phase4,2028.0,"FBP, EIR2 (land use change only)",25-T06-065
339,Interchange Improvements | US-101 | Produce Ave,Non-Transit Projects - Uncommitted: only in FB...,in net spec,SM110003_US101_ProduceAve,2040.0,"FBP, EIR2 (land use change only)",25-T06-054
569,I-80/I-680/SR-12 Interchange,PPA projects,in net spec,MAJ_SOL070020_I80_I680_SR12_Int_packages_3_4_5,NaN,"FBP, EIR2 (land use change only)",25-T06-024


['EXP_SC_SR85_SR87_to_US101_Phase4', 'ALA110002_I880_Industrial_Interchange', 'ALA170002_I80_Ashby_Int', 'FBP_SL_033_FairgroundsWide', 'FBP_CC_045_SanPabloDam_Interchange_Phase2', 'FBP_SC_042_I280_Downtown_Access_Improvements', 'FBP_SC_047_I280_Winchester_OffRamp', 'ALA250210_I580_Greenville', 'FBP_AL_039_I580_Interchange_Imps', 'ALA230222_I580_Vasco', 'FBP_SCL_050_I680_MontagueExpwy', 'FBP_CC_040_041_042_I680_SR4_Int_Phases_1_2_4_5', 'FBP_AL_042_I680_Stoneridge_Widening', 'FBP_AL_043_A_StreetWide', 'FBP_AL_045_Oak_Ala_Access_Pr', 'FBP_AL_044_I880_Whipple_Imps', 'FBP_ALA_I880_42nd_High_Interchange', 'FBP_SC_076_US101_Taylor_Interchange', 'SCL190012_US101_SanAntonio_Charleston_Rengstroff_Int', 'FBP_SM_033_US101_Holly_Interchange', 'SM110003_US101_ProduceAve', 'STIP_17_06_0010_WoodsideRd', 'FBP_SM_027_US101_92', 'FBP_SM_027_US101_92_directConnector', 'SCL210002_US101_SR152_Interchange', 'FBP_SC_083_US101_Zanker_Skyport_Interchange', 'FBP_SM_042_Hwy1_ManorDrive', 'FBP_SC_041_Envision_High

In [5]:
# projects for strategy T7
projects_T7_df = project_blueprint_df.loc[project_blueprint_df['RTP25_ID'].str.startswith('25-T07-')].copy()
print(projects_T7_df.shape[0], "projects in Strategy T7")
display(projects_T7_df.head(3))

# similarly, clean up to only keep the new name
has_newline = projects_T7_df['NetworkProjects Folder Name'].str.contains(' (', regex=False, na=False)
if has_newline.any():
    print('need to fix the following NetworkProjects Folder Name entries')
    display(projects_T7_df.loc[has_newline])
    projects_T7_df.loc[has_newline, 'NetworkProjects Folder Name'] = (
        projects_T7_df.loc[has_newline, 'NetworkProjects Folder Name'].str.split('\n').str[1]
    )
    projects_T7_df['NetworkProjects Folder Name'] = (
        projects_T7_df['NetworkProjects Folder Name'].str.replace(r' \(new\)$', '', regex=True)
    )
    print('fixed the following NetworkProjects Folder Name entries')
    display(projects_T7_df.loc[has_newline])

# print out the NetworkProjects Folder Name to add to net_spec_blueprint.py
print(projects_T7_df['NetworkProjects Folder Name'].tolist())

10 projects in Strategy T7


,Name,Section/Column,Coding Status,NetworkProjects Folder Name,Opening Year,In PBA50+,RTP25_ID
282,Express Lane | HOV Conversion/New Lane | I-680...,Non-Transit Projects - Uncommitted: only in FB...,in net spec,EXP_CC_I680NB_Livorna_to_Arthur,2030.0,"FBP, EIR2 (land use change only), Exclude from...",25-T07-075
289,Express Lane | New Lane | I-680 (NB) | Between...,Non-Transit Projects - Uncommitted: only in FB...,in net spec,EXP_AL_I680NB_SR84_to_Alcosta,2030.0,"FBP, EIR2 (land use change only), Exclude from...",25-T07-074
291,Express Lanes | HOV Conversion and New Lane | ...,Non-Transit Projects - Uncommitted: only in FB...,in net spec,EXP_SC_US101_SR237_to_I880_Phase5,2030.0,"FBP, EIR2 (land use change only), Exclude from...",25-T07-078


['EXP_CC_I680NB_Livorna_to_Arthur', 'EXP_AL_I680NB_SR84_to_Alcosta', 'EXP_SC_US101_SR237_to_I880_Phase5', 'EXP_SC_US101_I880_to_SR85_Phase6', 'EXP_SM_US101_I380_to_SFcounty', 'Transform_SR37_Widening_Interim', 'ALA090020_I880_Industrial_Parkway_AuxLanes', 'FBP_CC_050_SR4_Operation_Improvements_EB', 'FBP_CC_051_SR4_Operation_Improvements_WB', 'FBP_SC_054_SR17_Corridor_Relief']


In [6]:
# projects for strategy T8
projects_T8_df = project_blueprint_df.loc[project_blueprint_df['RTP25_ID'].str.startswith('25-T08-')].copy()
print(projects_T8_df.shape[0], "projects in Strategy T8")
display(projects_T8_df.head(3))

# some projects both '(old)' and '(new)' in NetworkProjects Folder Name, with old names coming from previous network coding;
# clean them up to only keep the new name
has_newline = projects_T8_df['NetworkProjects Folder Name'].str.contains(' (', regex=False, na=False)
if has_newline.any():
    print('need to fix the following NetworkProjects Folder Name entries')
    display(projects_T8_df.loc[has_newline][['Name','NetworkProjects Folder Name']])
    projects_T8_df.loc[has_newline, 'NetworkProjects Folder Name'] = (
        projects_T8_df.loc[has_newline, 'NetworkProjects Folder Name'].str.split("Phas").str[0]
    )
    projects_T8_df['NetworkProjects Folder Name'] = (
        projects_T8_df['NetworkProjects Folder Name'].str.replace(r' \($', '', regex=True)
    )
    print('fixed the following NetworkProjects Folder Name entries')
    display(projects_T8_df.loc[has_newline][['Name','NetworkProjects Folder Name']])

# print out the NetworkProjects Folder Name to add to net_spec_blueprint.py
print(projects_T8_df['NetworkProjects Folder Name'].tolist())

31 projects in Strategy T8


,Name,Section/Column,Coding Status,NetworkProjects Folder Name,Opening Year,In PBA50+,RTP25_ID
249,Roadway Widening | Jepson Pkwy-Leisure Town Rd...,Committed but Not Yet Under Construction: Add ...,in net spec,SOL110006_Jepson_1B_1C,2030.0,"NoProject, Exclude from EIR1 (transportation),...",25-T08-129
357,Roadway Extension | Caufield Ln | Between Baut...,Non-Transit Projects - Uncommitted: only in FB...,in net spec,FBP_SON_Caulfield_Extension,2040.0,"FBP, EIR2 (land use change only), Exclude from...",25-T08-133
358,Roadway Extension | Dublin Blvd-N Canyons Pkwy...,Non-Transit Projects - Uncommitted: only in FB...,in net spec,FBP_AL_055_DubBlvd_NCanyons_Ext,2040.0,"FBP, EIR2 (land use change only), Exclude from...",25-T08-096


need to fix the following NetworkProjects Folder Name entries


,Name,NetworkProjects Folder Name
379,"Roadway Widening | Brentwood Blvd, Phase 1 | B...",CC_070011_170015_Brentwood_Blvd_Widening (Phas...
380,"Roadway Widening | Brentwood Blvd, Phase 2 | B...",CC_070011_170015_Brentwood_Blvd_Widening (Phas...


fixed the following NetworkProjects Folder Name entries


,Name,NetworkProjects Folder Name
379,"Roadway Widening | Brentwood Blvd, Phase 1 | B...",CC_070011_170015_Brentwood_Blvd_Widening
380,"Roadway Widening | Brentwood Blvd, Phase 2 | B...",CC_070011_170015_Brentwood_Blvd_Widening


['SOL110006_Jepson_1B_1C', 'FBP_SON_Caulfield_Extension', 'FBP_AL_055_DubBlvd_NCanyons_Ext', 'SOL110004_Jepson_Walters_Extension', 'FBP_NP_066_Newell_Dr', 'FBP_SC_082_US101_25_Interchange', 'FBP_CC_061_062_West_Leland_Ext_Phases1_2', 'FBP_SC_088_Envision_Expwy', 'SCL210026_Julian_James_Conversion', 'FBP_NP_074_SoscolWide', 'FBP_NP_051_Airport_Junction', 'FBP_AL_052_AutoMallWide', 'CC_070011_170015_Brentwood_Blvd_Widening', 'CC_070011_170015_Brentwood_Blvd_Widening', 'FBP_SC_101_BrokawBridgeWide', 'FBP_SC_102_CalaverasWide', 'FBP_CC_064_CaminoTassajaraWide', 'FBP_CC_054_CrowCanyonWide', 'FBP_SF_090004_Harney_Wide', 'FBP_SL_042_Jepson_2B_2C', 'FBP_CC_057_LoneTreeWide', 'FBP_SC_103_MontagueWide', 'FBP_SC_104_OaklandWide', 'FBP_SL_053_PeabodyWide', 'FBP_CC_059_PittAntiochWide', 'FBP_AL_062_TassajaraWide', 'SCL250204_10thBridge_Ext', 'FBP_CC_067_WillowPassWide', 'RRSP_East_West_Connector', 'ALA210024_GrandAve_Improvements', 'MAJ_SR_239']


In [7]:
# projects for strategy T11
projects_T11_df = project_blueprint_df.loc[project_blueprint_df['RTP25_ID'].str.startswith('25-T11-')].copy()
print(projects_T11_df.shape[0], "projects in Strategy T11")
display(projects_T11_df.head(3))

# check and do similar clean up if needed
has_newline = projects_T11_df['NetworkProjects Folder Name'].str.contains('(old)', regex=False, na=False)
if has_newline.any():
    print('need to fix the following NetworkProjects Folder Name entries')
    display(projects_T11_df.loc[has_newline][['Name','NetworkProjects Folder Name']])
    projects_T11_df.loc[has_newline, 'NetworkProjects Folder Name'] = (
        projects_T11_df.loc[has_newline, 'NetworkProjects Folder Name'].str.split('\n').str[1]
    )
    projects_T11_df['NetworkProjects Folder Name'] = (
        projects_T11_df['NetworkProjects Folder Name'].str.replace(r' \(new\)$', '', regex=True)
    )
    print('fixed the following NetworkProjects Folder Name entries')
    display(projects_T11_df.loc[has_newline][['Name','NetworkProjects Folder Name']])

# print out the NetworkProjects Folder Name to add to net_spec_blueprint.py
print(projects_T11_df['NetworkProjects Folder Name'].tolist())

34 projects in Strategy T11


,Name,Section/Column,Coding Status,NetworkProjects Folder Name,Opening Year,In PBA50+,RTP25_ID
405,US 101 Bus on Shoulder Lanes,Small Projects,in net spec,FBP_MR_018_US101_BOS,2030.0,"FBP, EIR2 (land use change only)",25-T11-149
406,SF: Bayview Caltrain Station,Small Projects,in net spec,FBP_SF_Caltrain_Bayview,2030.0,"FBP, EIR2 (land use change only)",25-T11-171
408,NVTA Service Frequency Boost,Small Projects,review,FBP_NP_NapaVine_Serv_Freq_Boost,2026.0,"FBP, EIR2 (land use change only)",25-T11-151


['FBP_MR_018_US101_BOS', 'FBP_SF_Caltrain_Bayview', 'FBP_NP_NapaVine_Serv_Freq_Boost', 'FBP_CC_019_CCCTA_Freq_Increase', 'FBP_NP_040_VINE_Exp_Bus_Enhancements', 'FBP_SL_026_SolExpressBus', 'FBP_CC_028_Hercules_Station', 'FBP_SF_012_Geneva_Harney_BRT', 'MAJ_ElCaminoReal_BRT', 'FBP_MU_049_Caltrain_6TPHPD; FBP_MU_049_Caltrain_8TPHPD', 'RRSP_Alameda_Point_Transit_Improvements', 'MAJ_SC_VTAVisionaryNetwork', 'FBP_MU_046_ACE_Freq_Inc', 'FBP_MU_029_ACRapid_2025, FBP_MU_029_ACRapid_2040', 'MAJ_MTC050027_Berkeley_Ferry', 'MAJ_SanPablo_BRT', 'Dumbarton_Bridge_Bus_Service', 'MAJ_MuniForward_Uncommitted ', 'Transform_I680_Multimodal_Imp', 'FBP_SM_020_Regional_Express_Buses', 'GGT_Service_Imp', 'FBP_AL_021_South_Bay_Connect', 'MAJ_RedwoodCity_SF_Ferry', 'MAJ_WETA_Service_Frequency_Increase', 'FBP_CC_15_23rd_St_BRT', 'FBP_CC_018_BRT_Brentwood', 'MAJ_Sonoma_Frequency_Increase', 'FBP_MuniForward_Uncommitted_Rail', 'RRSP_South_East_Waterfront_Transit_Imp', 'MAJ_AC_Frequency_Improvement', 'RRSP_E14_Miss

In [8]:
# projects for strategy T12
projects_T12_df = project_blueprint_df.loc[project_blueprint_df['RTP25_ID'].str.startswith('25-T12-')].copy()
print(projects_T12_df.shape[0], "projects in Strategy T12")
display(projects_T12_df.head(3))

# check and do similar cleanup
has_newline = projects_T12_df['NetworkProjects Folder Name'].str.contains('),', regex=False, na=False)
if has_newline.any():
    print('need to fix the following NetworkProjects Folder Name entries')
    display(projects_T12_df.loc[has_newline][['Name','NetworkProjects Folder Name']])
    print(projects_T12_df.loc[has_newline]['NetworkProjects Folder Name'].tolist())

# just fix it manually is faster than cleaning up the df

# print out the NetworkProjects Folder Name to add to net_spec_blueprint.py
print(projects_T12_df['NetworkProjects Folder Name'].tolist())



5 projects in Strategy T12


,Name,Section/Column,Coding Status,NetworkProjects Folder Name,Opening Year,In PBA50+,RTP25_ID
232,VTA Eastridge to BART Regional Connector,Under Construction: Add to Baseline (FOR NETWO...,in net spec,MAJ_SCL050009_VTA_Eastridge_Extension,NaN,NoProject,25-T12-179
416,SMART: Cloverdale Extension,Downscoping/Phasing Projects,in net spec,"SON_SMART_to_Healdsburg (2026), SON_SMART_to_C...",2043.0,"FBP, EIR2 (land use change only)","25-T12-181, 25-T12-182"
544,Valley Link Rail Initial Operation Segment,PPA projects,in net spec,Transform_Valley_Link,2036.0,"FBP, EIR2 (land use change only)",25-T12-177


need to fix the following NetworkProjects Folder Name entries


,Name,NetworkProjects Folder Name
416,SMART: Cloverdale Extension,"SON_SMART_to_Healdsburg (2026), SON_SMART_to_C..."


['SON_SMART_to_Healdsburg (2026), SON_SMART_to_Cloverdale (2034)']
['MAJ_SCL050009_VTA_Eastridge_Extension', 'SON_SMART_to_Healdsburg (2026), SON_SMART_to_Cloverdale (2034)', 'Transform_Valley_Link', 'MAJ_BRT030001_BART_to_SanJose', 'MAJ_SF_050002_Caltrain_Ext_TransbayTerminal']


### Land Use Inputs

In [9]:
# for Final Blueprint minus Land Use strategies, use NoProject land use, but with T5 cordon pricing and parking pricing

NoProject_landuse_dir = pathlib.Path(r'M:\Application\Model One\RTP2025\INPUT_DEVELOPMENT\LandUse_n_Popsyn\BAUS_NoProject_v10\2050')
FBP_landuse_dir = pathlib.Path(r'M:\Application\Model One\RTP2025\INPUT_DEVELOPMENT\LandUse_n_Popsyn\BAUS_FBP_v08\2050')

# load the two versions of data and inspect
tazData_NP = pd.read_csv(NoProject_landuse_dir / 'landuse'/ 'tazData.csv')
tazData_FBP = pd.read_csv(FBP_landuse_dir / 'landuse' / 'parking_strategy' / 'tazData_parkingStrategy_v01_wSFCordon_wTICordon.csv')
display(tazData_NP.head())
display(tazData_FBP.head())


,ZONE,DISTRICT,SD,COUNTY,TOTHH,HHPOP,TOTPOP,EMPRES,SFDU,MFDU,...,OPRKCST,AREATYPE,HSENROLL,COLLFTE,COLLPTE,TOPOLOGY,TERMINAL,ZERO,CORDON,CORDONCOST
0,1,1,1,1,58,87,98,40,1,60,...,796.41905,0,0.0,0.00000,0.0,3,5.79044,0,0,0
1,2,1,1,1,189,279,314,135,0,207,...,716.00000,0,0.0,0.00000,0.0,1,5.67161,0,0,0
2,3,1,1,1,299,443,507,230,9,300,...,784.83951,0,0.0,0.00000,0.0,1,5.73928,0,0,0
3,4,1,1,1,285,417,435,187,6,293,...,336.59762,0,0.0,0.00000,0.0,2,5.76875,0,0,0
4,5,1,1,1,706,1042,1168,530,22,731,...,314.01431,0,0.0,75.58963,0.0,1,5.34252,0,0,0


,ZONE,DISTRICT,SD,COUNTY,TOTHH,HHPOP,TOTPOP,EMPRES,SFDU,MFDU,...,OPRKCST,AREATYPE,HSENROLL,COLLFTE,COLLPTE,TOPOLOGY,TERMINAL,ZERO,CORDON,CORDONCOST
0,1,1,1,1,205,309,320,138,1,214,...,1139.286388,0,0.0,0.00000,0.0,3,5.88019,0,10,378
1,2,1,1,1,540,802,837,354,0,582,...,938.175675,0,0.0,0.00000,0.0,1,5.73860,0,10,378
2,3,1,1,1,620,924,988,449,9,640,...,1544.360413,0,0.0,0.00000,0.0,1,6.04119,0,10,378
3,4,1,1,1,672,990,1008,450,6,694,...,522.971687,0,0.0,0.00000,0.0,2,5.91346,0,10,378
4,5,1,1,1,1764,2619,2745,1205,22,1937,...,392.517887,0,0.0,82.65037,0.0,1,5.61411,0,10,378


In [ ]:
# apply parking pricing on base parking rates for NP
# do this instead of directly using the parking pricing from FBP tazData because the latter is based on FBP land use density results
subprocess.run(
    [
        sys.executable,
        r"X:\travel-model-one-master\utilities\taz-data-builder\updateParkingCostForParkingStrategy.py",
        r"M:\Application\Model One\RTP2025\INPUT_DEVELOPMENT\LandUse_n_Popsyn\BAUS_NoProject_v10\2050\landuse\tazData.csv",
        r"M:\Application\Model One\RTP2030\StrategyEvaluation\INPUTS\LandUse\2050_NP_wParkingStrategy_wCordons\tazData_NP_wParkingStrategy.csv",
        "FBP",
    ],
    check=True,
)

tazData_NP_tazData_parkingStrategy_df = pd.read_csv(
    r"M:\Application\Model One\RTP2030\StrategyEvaluation\INPUTS\LandUse\2050_NP_wParkingStrategy_wCordons\tazData_NP_wParkingStrategy.csv"
)
display(tazData_NP_tazData_parkingStrategy_df.head())

,ZONE,DISTRICT,SD,COUNTY,TOTHH,HHPOP,TOTPOP,EMPRES,SFDU,MFDU,...,OPRKCST,AREATYPE,HSENROLL,COLLFTE,COLLPTE,TOPOLOGY,TERMINAL,ZERO,CORDON,CORDONCOST
0,1,1,1,1,58,87,98,40,1,60,...,995.523812,0,0.0,0.00000,0.0,3,5.79044,0,0,0
1,2,1,1,1,189,279,314,135,0,207,...,895.000000,0,0.0,0.00000,0.0,1,5.67161,0,0,0
2,3,1,1,1,299,443,507,230,9,300,...,981.049387,0,0.0,0.00000,0.0,1,5.73928,0,0,0
3,4,1,1,1,285,417,435,187,6,293,...,420.747025,0,0.0,0.00000,0.0,2,5.76875,0,0,0
4,5,1,1,1,706,1042,1168,530,22,731,...,392.517887,0,0.0,75.58963,0.0,1,5.34252,0,0,0


In [11]:
# bring cordon pricing fields from FBP
tazData_NP_tazData_parkingCordonStrategy_df = tazData_NP_tazData_parkingStrategy_df.loc[:, ~tazData_NP_tazData_parkingStrategy_df.columns.isin(['CORDON', 'CORDONCOST'])].copy()
tazData_NP_tazData_parkingCordonStrategy_df = pd.merge(
    tazData_NP_tazData_parkingCordonStrategy_df,
    tazData_FBP[['ZONE', 'CORDON', 'CORDONCOST']],
    on='ZONE',
    how='outer'
)
assert(tazData_NP_tazData_parkingCordonStrategy_df.shape[0] == 1454)
display(tazData_NP_tazData_parkingCordonStrategy_df.head())

,ZONE,DISTRICT,SD,COUNTY,TOTHH,HHPOP,TOTPOP,EMPRES,SFDU,MFDU,...,OPRKCST,AREATYPE,HSENROLL,COLLFTE,COLLPTE,TOPOLOGY,TERMINAL,ZERO,CORDON,CORDONCOST
0,1,1,1,1,58,87,98,40,1,60,...,995.523812,0,0.0,0.00000,0.0,3,5.79044,0,10,378
1,2,1,1,1,189,279,314,135,0,207,...,895.000000,0,0.0,0.00000,0.0,1,5.67161,0,10,378
2,3,1,1,1,299,443,507,230,9,300,...,981.049387,0,0.0,0.00000,0.0,1,5.73928,0,10,378
3,4,1,1,1,285,417,435,187,6,293,...,420.747025,0,0.0,0.00000,0.0,2,5.76875,0,10,378
4,5,1,1,1,706,1042,1168,530,22,731,...,392.517887,0,0.0,75.58963,0.0,1,5.34252,0,10,378


In [ ]:
# write out
tazData_NP_tazData_parkingCordonStrategy_df.to_csv(
    r"M:\Application\Model One\RTP2030\StrategyEvaluation\INPUTS\LandUse\2050_NP_wParkingStrategy_wCordons\tazData_NP_wParkingStrategy_wSFCordon_wTICordon.csv",
    index=False
)

# also need to create a dbf version using X:\travel-model-one-master\utilities\taz-data-csv-to-dbf\taz-data-csv-to-dbf.R